# 07 — Demanda sísmica, fragilidad y variables de los modelos de daño
Usa el **champion de amplificación** (`06_ml_regresion`) para pasar de los registros a la sacudida en cada edificio de los 6 distritos:
1. Tabla F(T, Vs30) del champion, término de sitio de cada estación y su validación.
2. Kriging de la sacudida en roca por sismo y demanda Sa(Tb) por edificio en cada escenario (registrado, diseño, 0.25 g, 0.70 g y Norma E.030).
3. Daño esperado con las curvas de fragilidad GEM.
4. Feature engineering de los modelos que dependen de la amplificación: `fe_dano` (Tainan y Manta, con las mismas variables que hay en Lima) y `fe_clustering` (perfiles de riesgo de los edificios de Lima).
5. Validación externa en Turquía 2023 (si se cargó): sacudida del ShakeMap en cada edificio (`fe_dano_turquia`) y fragilidad GEM frente al daño observado, por edificio y por zonas (`riesgo_validacion_turquia`, `riesgo_validacion_turquia_zonas`).
6. Calibración del acelerograma sintético (método estocástico de Boore) con los registros de Lima, validada dejando fuera un sismo. Sus parámetros quedan en `riesgo_parametros` y `11_sismo_tiempo_real` solo los usa para simular.

## 1. Setup — imports

In [ ]:
%pip install -q "numpy<2" pyproj shapely seaborn

In [ ]:
# Reinicia Python para que el driver y los workers usen las mismas librerías.
dbutils.library.restartPython()

In [ ]:
# Librerías que usa el notebook.
import json
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from pyproj import Transformer
from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import curve_fit
from scipy.signal import cont2discrete, lfilter, ss2tf

from pyspark.sql import functions as F, types as T
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from pyspark.ml import PipelineModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

## 2. Catálogo, esquema y parámetros

In [ ]:
# Catálogo, esquema y zona horaria de la sesión.
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

In [ ]:
# Área de estudio y constantes del procesamiento de señales.
PROVINCIAS_ESTUDIO = ["LIMA", "CALLAO"]
DISTRITOS_ESTUDIO = {"150104": "BARRANCO", "150122": "MIRAFLORES", "150130": "SAN BORJA", "150131": "SAN ISIDRO",
                     "150140": "SANTIAGO DE SURCO", "150141": "SURQUILLO"}
ZONA_E030_LIMA = 4
EPSG_UTM_LIMA = 32718
SEED = 700
FILTRO_HZ = (0.1, 25.0)
AMORTIGUAMIENTO = 0.05
PERIODOS = np.round(np.geomspace(0.05, 4.0, 50), 4)
G_CM_S2 = 980.665

## 3. Funciones y referencias

In [ ]:
# Pasa una tabla pandas pequeña a Spark, con los nulos bien tratados.
def a_spark(pdf, esquema=None):
    pdf = pdf.copy()
    for c in pdf.columns:
        if isinstance(pdf[c].dtype, (pd.StringDtype, pd.ArrowDtype)):
            pdf[c] = pdf[c].astype(object)
        if pdf[c].dtype == object:
            pdf[c] = pdf[c].map(lambda v: v if v is None or isinstance(v, (str, bytes, list, dict, np.ndarray))
                                else (None if pd.isna(v) else str(v)))
    return spark.createDataFrame(pdf, esquema) if esquema is not None else spark.createDataFrame(pdf)


# Guarda un DataFrame como tabla Delta y cuenta sus filas.
def escribir(df, tabla, particion=None, modo="overwrite", contar=True):
    escritor = df.write.format("delta").mode(modo)
    if modo == "overwrite":
        escritor = escritor.option("overwriteSchema", "true")
    if particion:
        escritor = escritor.partitionBy(particion)
    escritor.saveAsTable(tabla)
    if contar:
        n = spark.table(tabla).count()
        print(f"  {tabla}: {n:,} filas")
        return n


# Filas y tamaño de las tablas con un prefijo.
def inventario(prefijo):
    filas = []
    for t in spark.sql("SHOW TABLES").collect():
        if t.isTemporary or not t.tableName.startswith(prefijo):
            continue
        d = spark.sql(f"DESCRIBE DETAIL {t.tableName}").first()
        filas.append((t.tableName, spark.table(t.tableName).count(), round((d["sizeInBytes"] or 0) / 1e6, 1)))
    return spark.createDataFrame(filas, "tabla string, filas long, mb double")

In [ ]:
try:
    spark.sql("SELECT st_contains(st_geomfromwkt('POLYGON((0 0,1 0,1 1,0 1,0 0))', 4326), st_point(0.5, 0.5, 4326)) AS ok").first()
    USAR_ST = True
except Exception:
    USAR_ST = False


# Caja envolvente de cada geometría WKT.
@F.pandas_udf("xmin double, ymin double, xmax double, ymax double")
def limites_wkt(wkt: pd.Series) -> pd.DataFrame:
    return pd.DataFrame(shapely.bounds(shapely.from_wkt(wkt.to_numpy(dtype=object))), columns=["xmin", "ymin", "xmax", "ymax"], index=wkt.index)


# Pasa lon y lat a UTM 18S.
@F.pandas_udf("x double, y double")
def utm_lima(lon: pd.Series, lat: pd.Series) -> pd.DataFrame:
    x, y = Transformer.from_crs(4326, EPSG_UTM_LIMA, always_xy=True).transform(lon.to_numpy(float), lat.to_numpy(float))
    return pd.DataFrame({"x": x, "y": y}, index=lon.index)


# Join espacial punto en polígono con funciones ST y filtro por caja.
def unir_poligonos(puntos, poligonos, atributos, clave, lon="lon", lat="lat"):
    pol = (poligonos.where(F.col("geometry_wkt").isNotNull()).select(*atributos, F.col("geometry_wkt").alias("_pol_wkt"))
           .withColumn("_pid", F.row_number().over(Window.orderBy(*atributos))))
    pts = puntos.select(*clave, F.col(lon).alias("_lon"), F.col(lat).alias("_lat"))
    if USAR_ST:
        pol = (pol.withColumn("_b", limites_wkt("_pol_wkt"))
               .select("*", *[F.col(f"_b.{c}").alias(f"_{c}") for c in ("xmin", "ymin", "xmax", "ymax")]).drop("_b"))
        cond = (F.col("_lon").between(F.col("_xmin"), F.col("_xmax")) & F.col("_lat").between(F.col("_ymin"), F.col("_ymax"))
                & F.expr("st_contains(st_geomfromwkt(_pol_wkt, 4326), st_point(_lon, _lat, 4326))"))
        cruces = pts.join(F.broadcast(pol), cond, "inner").select(*clave, "_pid", *atributos)
    else:
        filas = pol.collect()
        arbol = shapely.STRtree(shapely.from_wkt([f["_pol_wkt"] for f in filas]))
        pids = np.array([f["_pid"] for f in filas])
        valores = {c: [f[c] for f in filas] for c in atributos}
        esquema = T.StructType(list(pts.select(*clave).schema) + [T.StructField("_pid", T.IntegerType())] + list(pol.select(*atributos).schema))

        # Cruza puntos y polígonos con un índice espacial cuando no hay funciones ST.
        def cruzar(lotes):
            for pdf in lotes:
                i_pt, i_pol = arbol.query(shapely.points(pdf["_lon"].to_numpy(float), pdf["_lat"].to_numpy(float)), predicate="within")
                cruce = pdf.iloc[i_pt][clave].reset_index(drop=True)
                cruce["_pid"] = pids[i_pol]
                for c in atributos:
                    cruce[c] = [valores[c][k] for k in i_pol]
                yield cruce

        cruces = pts.mapInPandas(cruzar, esquema)
    primero = Window.partitionBy(*clave).orderBy("_pid")
    cruces = cruces.withColumn("_r", F.row_number().over(primero)).where("_r = 1").drop("_r", "_pid")
    return puntos.join(cruces, clave, "left")

In [ ]:
METRICAS = []


# Anota las métricas de un modelo para guardarlas al final.
def registrar_modelo(nombre_modelo, params, metricas):
    for k, v in metricas.items():
        METRICAS.append({"modelo": nombre_modelo, "metrica": k, "valor": float(v), "parametros": json.dumps(params, default=str)})
    print(f"  {nombre_modelo}: " + " | ".join(f"{k} = {v:,.4f}" for k, v in metricas.items()))


# Reemplaza las métricas del notebook en ml_metricas.
def guardar_metricas(notebook):
    pdf = pd.DataFrame(METRICAS, columns=["modelo", "metrica", "valor", "parametros"])
    pdf["fecha"] = datetime.now().isoformat(timespec="seconds")
    pdf["notebook"] = notebook
    (a_spark(pdf, "modelo string, metrica string, valor double, parametros string, fecha string, notebook string")
     .write.mode("overwrite").option("replaceWhere", f"notebook = '{notebook}'").saveAsTable("ml_metricas"))
    display(pdf)


# Raíz del error cuadrático medio.
def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))

In [ ]:
# Carga el modelo vigente de un caso y sus metadatos.
def champion(caso):
    fila = (spark.table("ml_registro_modelos").filter((F.col("caso") == caso) & F.col("promovido"))
            .orderBy(F.col("version").desc()).first())
    if fila is None:
        raise RuntimeError(f"No hay champion de {caso}: ejecute antes su notebook de ML")
    meta = {"version": int(fila["version"]), "algoritmo": fila["algoritmo"], "lote": fila["lote"],
            "cols_num": json.loads(fila["columnas_num"]), "metricas": json.loads(fila["metricas"]), "detalle": json.loads(fila["detalle"])}
    print(f"Champion de {caso}: v{meta['version']} ({meta['algoritmo']}, lote {meta['lote']})")
    return PipelineModel.load(fila["ruta"]), meta

In [ ]:
# Lee una referencia de gold como tabla pandas pequeña.
def referencia(tipo):
    return pd.DataFrame([json.loads(r.valor) for r in spark.table("gold_referencias").where(F.col("tipo") == tipo).select("valor").collect()])


# Lee una referencia de gold como DataFrame de Spark.
def referencia_sp(tipo, ddl):
    return (spark.table("gold_referencias").where(F.col("tipo") == tipo)
            .select(F.from_json("valor", ddl).alias("v")).select("v.*"))


REG_LIMA = spark.table("gold_registros").where("ciudad = 'Lima'")
E030 = {f["perfil"]: f for f in referencia("e030").to_dict("records")}
Z_DISENO = float(E030["S1"]["Z_g"])
DISTRITO_PDF = {"cercado_de_lima": "LIMA", "comas": "COMAS", "la_molina": "LA MOLINA", "villa_el_salvador": "VILLA EL SALVADOR"}
a_utm = Transformer.from_crs(4326, EPSG_UTM_LIMA, always_xy=True)
EVENTO_EFECTO = "2025-06-15_callao_M6.1"
PGA_EFECTO = [0.25, 0.70]
VS30_REF_MIN = 500.0
VS30_GRID = np.arange(150.0, 1001.0, 10.0)
COBERTURA_KM = 10.0


# Perfil de suelo E.030 según el Vs30.
def perfil_e030(vs30):
    return (F.when(vs30 > 1500, "S0").when(vs30 > 500, "S1").when(vs30 >= 180, "S2").otherwise("S3"))

## 4. Champion de amplificación

In [ ]:
# Champion de regresión y espectros con la partición de estaciones.
champion_amp, meta_amp = champion("regresion")
ALGORITMO_AMP, VERSION_AMP = meta_amp["algoritmo"], meta_amp["version"]
esp = (spark.table("fe_regresion")
       .select("evento", "archivo_estacion", "estacion", "lat", "lon", "vs30", "dist_hipo_km", "indice_periodo", "T", "Sa",
               "ln_sa_r", "ln_T", "ln_vs30", "perfil_e030"))
esp_split = esp.withColumn("conjunto", F.when(F.abs(F.xxhash64("estacion", F.lit(SEED))) % 5 == 0, "test").otherwise("train"))

## 5. Tabla F(T, Vs30) y término de sitio de cada estación

In [ ]:
L_SITIO_KM = 4.0
Z90 = 1.2816


# Interpola el término de sitio con pesos gaussianos, en numpy.
def interpolar_sitio(xy_dest, xy_est, delta_est):
    d = np.sqrt(((xy_dest[:, None, :] - xy_est[None, :, :]) ** 2).sum(-1))
    w = np.exp(-(d / L_SITIO_KM) ** 2)
    return (w @ delta_est) / (w.sum(1, keepdims=True) + 1.0)


grilla = pd.DataFrame({"vs30": np.repeat(VS30_GRID, len(PERIODOS)),
                       "indice_periodo": np.tile(np.arange(len(PERIODOS)), len(VS30_GRID)),
                       "T": np.tile(PERIODOS, len(VS30_GRID))})
grilla_sp = (a_spark(grilla).withColumn("ln_T", F.log("T")).withColumn("ln_vs30", F.log("vs30"))
             .withColumn("perfil_e030", perfil_e030(F.col("vs30"))))
f_grid = (champion_amp.transform(grilla_sp).select("vs30", "indice_periodo", "prediction").toPandas()
          .sort_values(["vs30", "indice_periodo"]))
ancla = (champion_amp.transform(esp).where(F.col("vs30") >= VS30_REF_MIN)
         .groupBy("indice_periodo").agg(F.avg("prediction").alias("ancla")).toPandas()
         .set_index("indice_periodo").ancla.reindex(range(len(PERIODOS))).interpolate().bfill().ffill())
ln_f = f_grid.prediction.to_numpy().reshape(len(VS30_GRID), len(PERIODOS)) - ancla.to_numpy()[None, :]
interp_ln_f = RegularGridInterpolator((VS30_GRID, np.log(PERIODOS)), ln_f)


# Logaritmo de la amplificación lineal F(T, Vs30), desde la tabla del champion.
def ln_amplificacion(T, vs30):
    T, vs30 = np.broadcast_arrays(np.asarray(T, float), np.asarray(vs30, float))
    puntos = np.column_stack([np.clip(vs30.ravel(), VS30_GRID[0], VS30_GRID[-1]),
                              np.log(np.clip(T.ravel(), PERIODOS[0], PERIODOS[-1]))])
    return interp_ln_f(puntos).reshape(T.shape)


# Versión UDF de la amplificación lineal.
@F.pandas_udf("double")
def ln_amp(periodo: pd.Series, vs30: pd.Series) -> pd.Series:
    return pd.Series(ln_amplificacion(periodo.to_numpy(float), vs30.to_numpy(float)), index=periodo.index)


# Agrega las coordenadas UTM en kilómetros.
def con_xy_km(df):
    return (df.withColumn("_xy", utm_lima("lon", "lat")).withColumn("x_km", F.col("_xy.x") / 1000.0)
            .withColumn("y_km", F.col("_xy.y") / 1000.0).drop("_xy"))


# Término de sitio por estación y periodo, y coordenadas de las estaciones.
def terminos_sitio_sp(res):
    deltas = res.groupBy("estacion", "indice_periodo").agg(F.avg("res_sitio").alias("delta"))
    estaciones = con_xy_km(res.groupBy("estacion").agg(F.min_by("lon", "evento").alias("lon"), F.min_by("lat", "evento").alias("lat")))
    completo = (estaciones.select("estacion").crossJoin(periodos_sp.select("indice_periodo")).join(deltas, ["estacion", "indice_periodo"], "left")
                .withColumn("delta", F.coalesce("delta", F.lit(0.0))))
    return completo, estaciones


# Interpola el término de sitio con pesos gaussianos, en Spark.
def interpolar_sitio_sp(destinos, clave, deltas, estaciones):
    d2 = (F.col("x_km") - F.col("ex_km")) ** 2 + (F.col("y_km") - F.col("ey_km")) ** 2
    pesos = (destinos.select(*clave, "x_km", "y_km")
             .crossJoin(F.broadcast(estaciones.select(F.col("estacion").alias("origen"), F.col("x_km").alias("ex_km"), F.col("y_km").alias("ey_km"))))
             .withColumn("w", F.exp(-d2 / L_SITIO_KM ** 2)))
    suma = pesos.groupBy(*clave).agg(F.sum("w").alias("suma_w"))
    return (pesos.join(F.broadcast(deltas.withColumnRenamed("estacion", "origen")), "origen").groupBy(*clave, "indice_periodo").agg(F.sum(F.col("w") * F.col("delta")).alias("num"))
            .join(suma, clave).withColumn("delta", F.col("num") / (F.col("suma_w") + 1)).select(*clave, "indice_periodo", "delta"))


# RMSE entre dos columnas como expresión de Spark.
def error_rmse(a, b):
    return F.sqrt(F.avg((F.col(a) - F.col(b)) ** 2))


periodos_sp = spark.createDataFrame([(i, float(p)) for i, p in enumerate(PERIODOS)], "indice_periodo long, T double")
escribir(esp_split.withColumn("f", ln_amp("T", "vs30")), "riesgo_tmp_espectros", contar=False)
esp_tab = spark.table("riesgo_tmp_espectros")

CLAVE_ETA = ["evento", "indice_periodo"]
tr = esp_tab.where(F.col("conjunto") == "train")
eta_tr = tr.groupBy(*CLAVE_ETA).agg(F.avg(F.col("ln_sa_r") - F.col("f")).alias("eta"))
res_tr = tr.join(eta_tr, CLAVE_ETA).withColumn("res_sitio", F.col("ln_sa_r") - F.col("eta") - F.col("f"))
deltas_tr, estaciones_tr = terminos_sitio_sp(res_tr)
te = esp_tab.where(F.col("conjunto") == "test").join(eta_tr, CLAVE_ETA)
est_te = con_xy_km(te.groupBy("estacion").agg(F.min_by("lon", "evento").alias("lon"), F.min_by("lat", "evento").alias("lat")))
delta_te = interpolar_sitio_sp(est_te, ["estacion"], deltas_tr, estaciones_tr)
medias = (te.join(delta_te, ["estacion", "indice_periodo"])
          .groupBy("estacion", "indice_periodo")
          .agg(F.avg(F.col("f") + F.col("delta")).alias("con_sitio"), F.avg("f").alias("sin_sitio"), F.avg(F.col("ln_sa_r") - F.col("eta")).alias("obs")))
sigmas = medias.agg(error_rmse("con_sitio", "obs").alias("con"), error_rmse("sin_sitio", "obs").alias("sin")).first()
SIGMA_F_SITIO, SIGMA_F_SIN_SITIO = float(sigmas.con), float(sigmas.sin)
print(f"σ de la amplificación en un sitio nuevo: {SIGMA_F_SITIO:.3f} en ln (sin término de sitio: {SIGMA_F_SIN_SITIO:.3f})")

eta_todo = esp_tab.groupBy(*CLAVE_ETA).agg(F.avg(F.col("ln_sa_r") - F.col("f")).alias("eta"))
res_todo = esp_tab.join(eta_todo, CLAVE_ETA).withColumn("res_sitio", F.col("ln_sa_r") - F.col("eta") - F.col("f"))
deltas_sitio, estaciones_sitio = terminos_sitio_sp(res_todo)

rango_vs30 = esp_tab.agg(F.min("vs30"), F.max("vs30")).first()
VS30_ENTRENO = (float(rango_vs30[0]), float(rango_vs30[1]))
amp = grilla.assign(ln_F=ln_f.ravel())
amp["F"] = np.exp(amp.ln_F)
amp["F_p10"] = np.exp(amp.ln_F - Z90 * SIGMA_F_SITIO)
amp["F_p90"] = np.exp(amp.ln_F + Z90 * SIGMA_F_SITIO)
amp["perfil_e030"] = np.select([amp.vs30 > 1500, amp.vs30 > 500, amp.vs30 >= 180], ["S0", "S1", "S2"], "S3")
amp["extrapolado"] = (amp.vs30 < VS30_ENTRENO[0]) | (amp.vs30 > VS30_ENTRENO[1])
escribir(a_spark(amp), "riesgo_amplificacion_lima")
F_GRID = amp.F.to_numpy().reshape(len(VS30_GRID), len(PERIODOS))

por_estacion = esp_tab.groupBy("estacion").agg(F.min_by("lon", "evento").alias("lon"), F.min_by("lat", "evento").alias("lat"),
                                               F.min_by("vs30", "evento").alias("vs30"), F.countDistinct("evento").alias("eventos"))
sitio_est = (deltas_sitio.withColumnRenamed("delta", "delta_ln").join(periodos_sp, "indice_periodo").join(por_estacion, "estacion")
             .select("estacion", "indice_periodo", "delta_ln", "T", "lon", "lat", "vs30", "eventos"))
escribir(sitio_est, "riesgo_amplificacion_sitio_estaciones")
sitio_tab = spark.table("riesgo_amplificacion_sitio_estaciones")

registrar_modelo(f"amplificacion_{ALGORITMO_AMP}_v{VERSION_AMP}", {"ancla": f"Vs30 >= {VS30_REF_MIN:.0f}", "validacion_sitio": "20 % de estaciones (hash)"},
                 {"sigma_ln_amplificacion_sitio_nuevo": SIGMA_F_SITIO, "sigma_ln_sin_termino_sitio": SIGMA_F_SIN_SITIO})
resumen_sitio = sitio_tab.agg(F.countDistinct("estacion"), F.avg(F.abs("delta_ln"))).first()
print(f"Término de sitio de {resumen_sitio[0]} estaciones: |δ| medio = {resumen_sitio[1]:.2f} en ln")

## 6. Validación de la amplificación

In [ ]:
# Compara la amplificación con la E.030 y la microzonificación, y la grafica.
meseta = amp[amp["T"] <= E030["S1"]["TP_s"]].groupby("perfil_e030").F.mean()
print("F medio en la meseta (T ≤ TP de S1):", meseta.round(3).to_dict())
if {"S1", "S2"} <= set(meseta.index):
    print(f"S2/S1 medido = {meseta['S2'] / meseta['S1']:.3f}   |   S2/S1 E.030 = {E030['S2']['S'] / E030['S1']['S']:.3f}")

est = REG_LIMA.select("evento", "archivo_estacion", "estacion", "lon", "lat", "tp_s", "vs30")
dist_sp = referencia_sp("distritos", "distrito string, geometry_wkt string")
est_d = unir_poligonos(est, dist_sp, ["distrito"], ["evento", "archivo_estacion"]).where(F.col("distrito").isin(list(DISTRITO_PDF.values())))
mz = (referencia_sp("microzonificacion_periodos", "distrito string, T_min_s double, T_max_s double").groupBy("distrito")
      .agg(F.min("T_min_s").alias("T_min_microzon_s"), F.max("T_max_s").alias("T_max_microzon_s")))
display(est_d.groupBy("distrito").agg(F.count("tp_s").alias("registros"), F.median("tp_s").alias("tp_mediano_s"), F.avg("vs30").alias("vs30_medio"))
        .join(mz, "distrito", "left"))

fig, ax = plt.subplots(figsize=(8, 4.5))
for v in [200, 300, 400, 500, 700, 900]:
    i = int(np.argmin(np.abs(VS30_GRID - v)))
    d = amp[amp.vs30 == VS30_GRID[i]]
    ax.semilogx(d["T"], d["F"], label=f"Vs30 = {VS30_GRID[i]:.0f} m/s")
    ax.fill_between(d["T"], d["F_p10"], d["F_p90"], alpha=.12)
ax.axhline(1, color="k", lw=.8)
ax.set(xlabel="Periodo T (s)", ylabel="F(T, Vs30)", title=f"Amplificación del suelo ({ALGORITMO_AMP} v{VERSION_AMP}, intervalo p10–p90 de un sitio nuevo)")
ax.legend(fontsize=8); ax.grid(True, which="both", alpha=.3)
plt.tight_layout(); plt.show()

## 7. Kriging de la sacudida en roca y escenarios

In [ ]:
COBERTURA_KM = 10.0
FACTOR_DISENO_MAX = 10.0
MIN_ESTACIONES_ESCENARIO = 10


# Variograma exponencial.
def variograma_exp(h, pepita, meseta, rango):
    return pepita + meseta * (1 - np.exp(-3 * h / rango))


# Ajusta un variograma exponencial; devuelve la pepita relativa y el rango.
def ajustar_variograma(xy, valores):
    d = np.sqrt(((xy[:, None, :] - xy[None, :, :]) ** 2).sum(-1))
    iu = np.triu_indices(len(xy), 1)
    h = d[iu]
    g = 0.5 * np.mean((valores[:, iu[0]] - valores[:, iu[1]]) ** 2, axis=0)
    rango0 = max(float(np.median(h)), 2.0)
    try:
        (pep, mes, rango), _ = curve_fit(variograma_exp, h, g, p0=[0.1, 1.0, rango0],
                                         bounds=([0.0, 0.01, 1.0], [2.0, 5.0, max(5 * h.max(), 2 * rango0)]))
    except Exception:
        pep, mes, rango = 0.1, 1.0, rango0
    return pep / (pep + mes), rango


# Pesos del kriging ordinario y, si se pide, su desviación relativa.
def pesos_kriging(xy, xy0, pepita_rel, rango, con_varianza=False):
    n = len(xy)
    gam = lambda h: np.where(h > 0, pepita_rel + (1 - pepita_rel) * (1 - np.exp(-3 * h / rango)), 0.0)
    A = np.ones((n + 1, n + 1))
    A[:n, :n] = gam(np.sqrt(((xy[:, None] - xy[None]) ** 2).sum(-1)))
    A[n, n] = 0.0
    B = np.ones((n + 1, len(xy0)))
    B[:n] = gam(np.sqrt(((xy[:, None] - xy0[None]) ** 2).sum(-1)))
    sol = np.linalg.lstsq(A, B, rcond=None)[0]
    if not con_varianza:
        return sol[:n].T
    varianza = (sol[:n] * B[:n]).sum(0) + sol[n]
    return sol[:n].T, np.sqrt(np.clip(varianza, 0, None))


sitio_pd = sitio_tab.select("estacion", "indice_periodo", "delta_ln", "lon", "lat").toPandas()
DELTA_SITIO = sitio_pd.pivot(index="estacion", columns="indice_periodo", values="delta_ln").sort_index().sort_index(axis=1)
coord_sitio = sitio_pd.groupby("estacion")[["lon", "lat"]].first().loc[DELTA_SITIO.index]
XY_SITIO = np.column_stack(a_utm.transform(coord_sitio.lon.to_numpy(), coord_sitio.lat.to_numpy())) / 1000.0

delta0 = sitio_tab.where(F.col("indice_periodo") == 0).select("estacion", F.col("delta_ln").alias("delta0"))
reg = (REG_LIMA.select("evento", "archivo_estacion", "estacion", "lon", "lat", "vs30", "pga_geo_g", "pgv_geo_cms").where("vs30 IS NOT NULL")
       .join(F.broadcast(delta0), "estacion", "left")
       .withColumn("pga_roca_g", F.col("pga_geo_g") / F.exp(ln_amp(F.lit(float(PERIODOS[0])), F.col("vs30"))) / F.exp(F.coalesce("delta0", F.lit(0.0))))
       .drop("delta0"))
PGA_ROCA = {r.evento: float(r.m) for r in reg.groupBy("evento").agg(F.median("pga_roca_g").alias("m")).collect()}
PGA_REF_NL = float(reg.agg(F.median("pga_roca_g")).first()[0])

celdas = con_xy_km(spark.table("gold_edificios").groupBy("h3_r8").agg(F.avg("lon").alias("lon"), F.avg("lat").alias("lat")))
delta_celdas = interpolar_sitio_sp(celdas, ["h3_r8"], deltas_sitio, estaciones_sitio)
sitio_h3 = (delta_celdas.groupBy("h3_r8").agg(F.array_sort(F.collect_list(F.struct("indice_periodo", "delta"))).alias("p"))
            .select("h3_r8", F.transform("p", lambda e: F.exp(e["delta"])).alias("dS_arr")))
escribir(sitio_h3, "riesgo_sitio_h3")
celdas_pd = celdas.select("h3_r8", "x_km", "y_km").orderBy("h3_r8").toPandas()
H3_CELDAS, xy_celdas = celdas_pd.h3_r8.to_numpy(dtype=object), celdas_pd[["x_km", "y_km"]].to_numpy(float)
media_delta = spark.table("riesgo_sitio_h3").select(F.explode("dS_arr").alias("x")).agg(F.avg(F.abs(F.log("x")))).first()[0]
print(f"Celdas H3 r8 con edificios: {len(celdas_pd):,} | corrección de sitio |δ| media = {media_delta:.2f} en ln")

COLS_KRIGING = ["h3_r8", "escenario", "evento", "tipo", "factor_escala", "sa_roca", "pgv_cms", "pga_roca_g", "kriging_sd_rel", "dist_estacion_km",
                "estaciones", "pga_roca_mediana_g", "pga_roca_escenario_g", "variograma_pepita_rel", "variograma_rango_km"]
ESQ_KRIGING = ("h3_r8 string, escenario string, evento string, tipo string, factor_escala double, sa_roca array<double>, pgv_cms double, pga_roca_g double, "
               "kriging_sd_rel double, dist_estacion_km double, estaciones long, pga_roca_mediana_g double, pga_roca_escenario_g double, "
               "variograma_pepita_rel double, variograma_rango_km double")
ESQ_LOSO = ("evento string, archivo_estacion string, indice_periodo long, ln_obs double, ln_kriging double, ln_media_otras double, ln_sa_obs double, "
            "ln_sa_cadena double, ln_sa_kriging_directo double")


# Matrices de un sismo y su variograma; None si tiene pocas estaciones.
def matrices_evento(d):
    piv = d.pivot_table(index="archivo_estacion", columns="indice_periodo", values="ln_sa_roca").dropna()
    if len(piv) < MIN_ESTACIONES_ESCENARIO:
        return None
    est_ev = d.drop_duplicates("archivo_estacion").set_index("archivo_estacion").loc[piv.index]
    transformador = Transformer.from_crs(4326, EPSG_UTM_LIMA, always_xy=True)
    xy = np.column_stack(transformador.transform(est_ev.lon.to_numpy(), est_ev.lat.to_numpy())) / 1000.0
    Z = piv.to_numpy()
    ln_sa = np.log(d.pivot_table(index="archivo_estacion", columns="indice_periodo", values="Sa").loc[piv.index].to_numpy())
    f_est = ln_amplificacion(np.tile(PERIODOS, len(xy)), np.repeat(est_ev.vs30.to_numpy(), len(PERIODOS))).reshape(len(xy), -1)
    pepita_rel, rango = ajustar_variograma(xy, ((Z - Z.mean(0)) / Z.std(0).clip(1e-6)).T)
    return piv, est_ev, xy, Z, ln_sa, f_est, pepita_rel, rango


# Se ejecuta por sismo: kriging en cada celda H3 para los escenarios registrado, de diseño y de efecto.
def escenarios_evento(d):
    matrices = matrices_evento(d)
    if matrices is None:
        return pd.DataFrame(columns=COLS_KRIGING)
    ev = d["evento"].iloc[0]
    piv, est_ev, xy, Z, _, _, pepita_rel, rango = matrices
    W, sd_rel = pesos_kriging(xy, xy_celdas, pepita_rel, rango, con_varianza=True)
    dist_min = np.sqrt(((xy_celdas[:, None] - xy[None]) ** 2).sum(-1)).min(1)
    sa_roca = np.exp(W @ Z)
    pgv = np.exp(W @ np.log(est_ev.pgv_geo_cms.to_numpy()))
    pga_r = np.exp(W @ np.log(est_ev.pga_roca_g.to_numpy()))
    casos = [("registrado", 1.0), ("diseno", Z_DISENO / PGA_ROCA[ev])] + ([(f"{p:.2f}g", p / PGA_ROCA[ev]) for p in PGA_EFECTO] if ev == EVENTO_EFECTO else [])
    partes = [pd.DataFrame({"h3_r8": H3_CELDAS, "escenario": f"{ev}_{tipo}", "evento": ev, "tipo": tipo, "factor_escala": k, "sa_roca": list(sa_roca * k),
                            "pgv_cms": pgv * k, "pga_roca_g": pga_r * k, "kriging_sd_rel": sd_rel, "dist_estacion_km": dist_min, "estaciones": len(piv),
                            "pga_roca_mediana_g": PGA_ROCA[ev], "pga_roca_escenario_g": PGA_ROCA[ev] * k,
                            "variograma_pepita_rel": pepita_rel, "variograma_rango_km": rango})
              for tipo, k in casos if not (tipo == "diseno" and k > FACTOR_DISENO_MAX)]
    return pd.concat(partes, ignore_index=True)


# Se ejecuta por sismo: validación dejando una estación afuera.
def loso_evento(d):
    matrices = matrices_evento(d)
    if matrices is None:
        return pd.DataFrame(columns=[c.split()[0] for c in ESQ_LOSO.split(", ")])
    ev = d["evento"].iloc[0]
    piv, est_ev, xy, Z, ln_sa, f_est, pepita_rel, rango = matrices
    partes = []
    for j in range(len(xy)):
        otras = np.arange(len(xy)) != j
        w_j = pesos_kriging(xy[otras], xy[j:j + 1], pepita_rel, rango)
        mascara = DELTA_SITIO.index != est_ev.estacion.iloc[j]
        delta_j = interpolar_sitio(xy[j:j + 1], XY_SITIO[mascara], DELTA_SITIO.to_numpy()[mascara])[0]
        roca_j = (w_j @ Z[otras])[0]
        partes.append(pd.DataFrame({"evento": ev, "archivo_estacion": piv.index[j], "indice_periodo": piv.columns.astype(int),
                                    "ln_obs": Z[j], "ln_kriging": roca_j, "ln_media_otras": Z[otras].mean(0),
                                    "ln_sa_obs": ln_sa[j], "ln_sa_cadena": roca_j + f_est[j] + delta_j,
                                    "ln_sa_kriging_directo": (w_j @ ln_sa[otras])[0]}))
    return pd.concat(partes, ignore_index=True)


datos_ev = (esp_tab.join(sitio_tab.select("estacion", "indice_periodo", "delta_ln"), ["estacion", "indice_periodo"])
            .withColumn("ln_sa_roca", F.log("Sa") - F.col("f") - F.col("delta_ln"))
            .select("evento", "archivo_estacion", "indice_periodo", "Sa", "ln_sa_roca")
            .join(reg.select("evento", "archivo_estacion", "estacion", "lon", "lat", "vs30", "pgv_geo_cms", "pga_roca_g"), ["evento", "archivo_estacion"]))
escribir(datos_ev.groupBy("evento").applyInPandas(escenarios_evento, ESQ_KRIGING), "riesgo_tmp_kriging", contar=False)
kriging = spark.table("riesgo_tmp_kriging")
escribir(kriging.select("h3_r8", "escenario", "evento", "tipo", "factor_escala", "sa_roca", "pgv_cms", "pga_roca_g", "kriging_sd_rel", "dist_estacion_km"), "riesgo_sa_roca_h3")
filas_norma = spark.createDataFrame(
    [(nombre, "E.030", tipo, 1.0, 0, Z_DISENO, Z_DISENO, None, None) for nombre, tipo in (("e030_diseno", "norma"), ("e030_sitio", "norma_sitio"))],
    "escenario string, evento string, tipo string, factor_escala double, estaciones long, pga_roca_mediana_g double, pga_roca_escenario_g double, "
    "variograma_pepita_rel double, variograma_rango_km double")
escribir(kriging.select(*filas_norma.columns).dropDuplicates(["escenario"]).unionByName(filas_norma), "riesgo_escenarios")
escribir(datos_ev.groupBy("evento").applyInPandas(loso_evento, ESQ_LOSO), "riesgo_kriging_validacion")
sin_escenario = sorted(set(PGA_ROCA) - {r.evento for r in spark.table("riesgo_escenarios").select("evento").distinct().collect()})
print(f"Sin escenario espacial (menos de {MIN_ESTACIONES_ESCENARIO} estaciones o factor de diseño mayor a {FACTOR_DISENO_MAX:.0f}):", sin_escenario)

loso = spark.table("riesgo_kriging_validacion")
display(loso.groupBy("evento").agg(F.countDistinct("archivo_estacion").alias("estaciones"), F.round(error_rmse("ln_kriging", "ln_obs"), 3).alias("rmse_ln_kriging_roca"),
                                   F.round(error_rmse("ln_media_otras", "ln_obs"), 3).alias("rmse_ln_media_roca"),
                                   F.round(error_rmse("ln_sa_cadena", "ln_sa_obs"), 3).alias("rmse_ln_sa_cadena"),
                                   F.round(error_rmse("ln_sa_kriging_directo", "ln_sa_obs"), 3).alias("rmse_ln_sa_kriging_directo"),
                                   F.round(F.avg(F.col("ln_sa_cadena") - F.col("ln_sa_obs")), 3).alias("sesgo_ln_sa_cadena")))
errores = loso.agg(error_rmse("ln_kriging", "ln_obs").alias("k"), error_rmse("ln_media_otras", "ln_obs").alias("m"),
                   error_rmse("ln_sa_cadena", "ln_sa_obs").alias("c"), error_rmse("ln_sa_kriging_directo", "ln_sa_obs").alias("d")).first()
SIGMA_LOSO = float(errores.c)
registrar_modelo("kriging_ordinario", {"variograma": "exponencial", "validacion": "leave-one-station-out"},
                 {"rmse_ln_kriging": errores.k, "rmse_ln_media_otras": errores.m})
registrar_modelo("demanda_cadena_completa", {"validacion": "leave-one-station-out", "cadena": "roca krigeada + F(T, Vs30) + termino de sitio interpolado"},
                 {"rmse_ln_sa_cadena": errores.c, "rmse_ln_sa_kriging_directo": errores.d})
display(spark.table("riesgo_sa_roca_h3").where(F.col("tipo") == "registrado").groupBy("evento")
        .agg(F.round(100 * F.avg((F.col("dist_estacion_km") > COBERTURA_KM).cast("double"))).alias(f"pct_celdas_a_mas_de_{COBERTURA_KM:.0f}_km")))
display(spark.table("riesgo_escenarios"))
for tmp in ["riesgo_tmp_kriging", "riesgo_tmp_espectros"]:
    spark.sql(f"DROP TABLE IF EXISTS {tmp}")

## 8. Demanda Sa(Tb) en cada edificio

In [ ]:
tb = np.array([r["Tb"] for r in spark.table("gold_edificios").select("Tb").distinct().collect()], float)
i = np.clip(np.searchsorted(PERIODOS, tb) - 1, 0, len(PERIODOS) - 2)
w = np.clip((np.log(tb) - np.log(PERIODOS[i])) / (np.log(PERIODOS[i + 1]) - np.log(PERIODOS[i])), 0, 1)
tb_tab = a_spark(pd.DataFrame({"Tb": tb, "idx": i + 1, "w": w}), "Tb double, idx int, w double")
amp_arr = a_spark(pd.DataFrame({"vs30_bin": VS30_GRID, "F_arr": [fila.tolist() for fila in F_GRID]}), "vs30_bin double, F_arr array<double>")
sitio_arr = spark.table("riesgo_sitio_h3")
F_SITIO = F.expr("zip_with(F_arr, coalesce(dS_arr, array_repeat(CAST(1.0 AS DOUBLE), size(F_arr))), (a, b) -> a * b)")
nl = referencia("bssa14").astype(float)
nl = nl[nl.T_s > 0].sort_values("T_s")
F4_NL = np.interp(np.log(PERIODOS), np.log(nl.T_s), nl.f4)
F5_NL = np.interp(np.log(PERIODOS), np.log(nl.T_s), nl.f5)
arr_sql = lambda v: "array(" + ", ".join(f"{x:.6f}D" for x in v) + ")"
F_NO_LINEAL = F.expr(f"transform(F_arr, (x, i) -> x * exp({arr_sql(F4_NL)}[i] * (exp({arr_sql(F5_NL)}[i] * (least(vs30, 760D) - 360D)) "
                     f"- exp({arr_sql(F5_NL)}[i] * 400D)) * (ln((pga_r_nl + 0.1D) / 0.1D) - {np.log((PGA_REF_NL + 0.1) / 0.1):.6f}D)))")
f2_250 = F4_NL * (np.exp(F5_NL * (250 - 360)) - np.exp(F5_NL * 400))
print(f"Suelo no lineal (BSSA14): PGA en roca de los registros = {PGA_REF_NL:.3f} g | con 0.45 g, Vs30 = 250 m/s amplifica "
      f"×{np.exp(f2_250[0] * (np.log(0.55 / 0.1) - np.log((PGA_REF_NL + 0.1) / 0.1))):.2f} a 0.05 s y "
      f"×{np.exp(np.interp(np.log(0.3), np.log(PERIODOS), f2_250) * (np.log(0.55 / 0.1) - np.log((PGA_REF_NL + 0.1) / 0.1))):.2f} a 0.3 s")


# Interpola en logaritmo un arreglo de Spark en el periodo de cada edificio.
def interp_log(arreglo):
    return F.exp((1 - F.col("w")) * F.log(F.expr(f"element_at({arreglo}, idx)"))
                 + F.col("w") * F.log(F.expr(f"element_at({arreglo}, idx + 1)")))


# Interpola en logaritmo un arreglo de Spark en un periodo fijo.
def interp_fijo(arreglo, T):
    j = int(np.clip(np.searchsorted(PERIODOS, T) - 1, 0, len(PERIODOS) - 2))
    wj = float(np.clip((np.log(T) - np.log(PERIODOS[j])) / (np.log(PERIODOS[j + 1]) - np.log(PERIODOS[j])), 0, 1))
    return F.exp((1 - wj) * F.log(F.expr(f"element_at({arreglo}, {j + 1})")) + wj * F.log(F.expr(f"element_at({arreglo}, {j + 2})")))


# Sa en el sitio para un periodo fijo.
def sa_sitio(T):
    return interp_fijo("sa_roca", T) * interp_fijo("F_arr", T)


COLS_DEMANDA = ["id", "escenario", "evento", "tipo", "ubigeo", "distrito", "provincia", "h3_id", "h3_r8", "pisos", "Tb", "TP",
                "Tb_TP", "vs30", "tipo_suelo", "Sa_roca_Tb", "F_Tb", "Sa_Tb", "pgv_cms", "PGA_g", "Sa03_g", "Sa06_g", "Sa10_g",
                "kriging_sd_rel", "dist_estacion_km", "fuera_cobertura"]
ed = (spark.table("gold_edificios")
      .select("id", "ubigeo", "distrito", "provincia", "h3_id", "h3_r8", "pisos", "Tb", "TP", "TL", "S", "Tb_TP", "vs30", "tipo_suelo")
      .where("vs30 IS NOT NULL")
      .withColumn("vs30_bin", F.least(F.lit(float(VS30_GRID.max())),
                                      F.greatest(F.lit(float(VS30_GRID.min())), F.round(F.col("vs30") / 10) * 10))))
demanda = (ed.join(F.broadcast(tb_tab), "Tb").join(F.broadcast(amp_arr), "vs30_bin")
           .join(F.broadcast(sitio_arr), "h3_r8", "left").withColumn("F_arr", F_SITIO)
           .join(spark.table("riesgo_sa_roca_h3"), "h3_r8")
           .withColumn("pga_r_nl", F.col("pga_roca_g")).withColumn("F_arr", F_NO_LINEAL)
           .withColumn("Sa_roca_Tb", interp_log("sa_roca"))
           .withColumn("F_Tb", interp_log("F_arr"))
           .withColumn("Sa_Tb", F.col("Sa_roca_Tb") * F.col("F_Tb"))
           .withColumn("PGA_g", F.col("pga_roca_g") * F.expr("element_at(F_arr, 1)"))
           .withColumn("Sa03_g", sa_sitio(0.3)).withColumn("Sa06_g", sa_sitio(0.6)).withColumn("Sa10_g", sa_sitio(1.0))
           .withColumn("fuera_cobertura", F.col("dist_estacion_km") > COBERTURA_KM)
           .select(*COLS_DEMANDA))


# Factor de amplificación C de la Norma E.030 para el perfil del edificio.
def c_e030(T):
    return (F.when(T < F.col("TP"), F.lit(2.5)).when(T < F.col("TL"), 2.5 * F.col("TP") / T)
             .otherwise(2.5 * F.col("TP") * F.col("TL") / (T * T)))


U_VIVIENDA = 1.0
zus = Z_DISENO * U_VIVIENDA * F.col("S")
demanda_e030 = (ed.withColumn("escenario", F.lit("e030_diseno")).withColumn("evento", F.lit("E.030")).withColumn("tipo", F.lit("norma"))
                .withColumn("Sa_roca_Tb", Z_DISENO * U_VIVIENDA * c_e030(F.col("Tb"))).withColumn("F_Tb", F.col("S"))
                .withColumn("Sa_Tb", F.col("Sa_roca_Tb") * F.col("S")).withColumn("pgv_cms", F.lit(None).cast("double"))
                .withColumn("PGA_g", zus).withColumn("Sa03_g", zus * c_e030(F.lit(0.3)))
                .withColumn("Sa06_g", zus * c_e030(F.lit(0.6))).withColumn("Sa10_g", zus * c_e030(F.lit(1.0)))
                .withColumn("kriging_sd_rel", F.lit(None).cast("double")).withColumn("dist_estacion_km", F.lit(None).cast("double"))
                .withColumn("fuera_cobertura", F.lit(False))
                .select(*COLS_DEMANDA))

S1 = E030["S1"]


# Factor C de la Norma E.030 para el perfil S1 (roca).
def c_roca(T):
    if isinstance(T, (int, float)):
        return 2.5 if T < S1["TP_s"] else (2.5 * S1["TP_s"] / T if T < S1["TL_s"] else 2.5 * S1["TP_s"] * S1["TL_s"] / T ** 2)
    return (F.when(T < S1["TP_s"], F.lit(2.5)).when(T < S1["TL_s"], 2.5 * S1["TP_s"] / T)
             .otherwise(2.5 * S1["TP_s"] * S1["TL_s"] / (T * T)))


zus1 = Z_DISENO * U_VIVIENDA * S1["S"]
demanda_sitio = (ed.join(F.broadcast(tb_tab), "Tb").join(F.broadcast(amp_arr), "vs30_bin")
                 .join(F.broadcast(sitio_arr), "h3_r8", "left").withColumn("F_arr", F_SITIO)
                 .withColumn("pga_r_nl", F.lit(float(Z_DISENO * S1["S"]))).withColumn("F_arr", F_NO_LINEAL)
                 .withColumn("escenario", F.lit("e030_sitio")).withColumn("evento", F.lit("E.030")).withColumn("tipo", F.lit("norma_sitio"))
                 .withColumn("Sa_roca_Tb", zus1 * c_roca(F.col("Tb"))).withColumn("F_Tb", interp_log("F_arr"))
                 .withColumn("Sa_Tb", F.col("Sa_roca_Tb") * F.col("F_Tb")).withColumn("pgv_cms", F.lit(None).cast("double"))
                 .withColumn("PGA_g", zus1 * F.expr("element_at(F_arr, 1)"))
                 .withColumn("Sa03_g", zus1 * c_roca(0.3) * interp_fijo("F_arr", 0.3))
                 .withColumn("Sa06_g", zus1 * c_roca(0.6) * interp_fijo("F_arr", 0.6))
                 .withColumn("Sa10_g", zus1 * c_roca(1.0) * interp_fijo("F_arr", 1.0))
                 .withColumn("kriging_sd_rel", F.lit(None).cast("double")).withColumn("dist_estacion_km", F.lit(None).cast("double"))
                 .withColumn("fuera_cobertura", F.lit(False))
                 .select(*COLS_DEMANDA))
demanda = demanda.unionByName(demanda_e030).unionByName(demanda_sitio)
n_demanda = escribir(demanda, "riesgo_demanda", particion="escenario")
(spark.table("riesgo_demanda").groupBy("escenario")
 .agg(F.count("*").alias("edificios"), F.round(F.expr("percentile_approx(Sa_Tb, 0.5)"), 3).alias("Sa_Tb_mediana_g"),
      F.round(F.max("Sa_Tb"), 3).alias("Sa_Tb_max_g"), F.round(F.avg("F_Tb"), 3).alias("F_Tb_medio"),
      F.round(F.avg("pgv_cms"), 2).alias("pgv_medio_cms"))
 .orderBy("escenario").show(truncate=False))

## 9. Daño esperado con las curvas de fragilidad GEM

In [ ]:
PISOS_CONCRETO = 5
PISOS_MAX_LIVIANO = 2
ESTADOS = ["leve", "moderado", "severo", "colapso"]
frag_pdf = referencia("fragilidad")
escribir(a_spark(frag_pdf), "riesgo_fragilidad_parametros")
PISOS_MAX_GEM = {t: int(v) for t, v in frag_pdf.groupby("tipologia").pisos_gem.max().items()}


# Función de distribución normal estándar como expresión de Spark.
def phi(x):
    z = F.abs(x) / np.sqrt(2.0)
    t = 1 / (1 + 0.3275911 * z)
    poli = t * (0.254829592 + t * (-0.284496736 + t * (1.421413741 + t * (-1.453152027 + t * 1.061405429))))
    erf = 1 - poli * F.exp(-z * z)
    return F.when(x >= 0, 0.5 * (1 + erf)).otherwise(0.5 * (1 - erf))


censo = F.broadcast(spark.table("gold_edificios").select("ubigeo", "p_adobe", "p_quincha", "p_madera").distinct())
u = (F.abs(F.xxhash64("id")) % 1_000_000) / 1_000_000.0
liviano = F.col("pisos") <= PISOS_MAX_LIVIANO
tipologia = (F.when(F.col("pisos") >= PISOS_CONCRETO, "concreto_armado")
              .when(liviano & (u < F.col("p_adobe")), "adobe")
              .when(liviano & (u < F.col("p_adobe") + F.col("p_quincha")), "quincha")
              .when(liviano & (u < F.col("p_adobe") + F.col("p_quincha") + F.col("p_madera")), "madera")
              .otherwise("albanileria_confinada"))
tope_pisos = F.lit(PISOS_MAX_GEM["albanileria_confinada"])
for tip, tope in PISOS_MAX_GEM.items():
    tope_pisos = F.when(F.col("tipologia") == tip, F.lit(tope)).otherwise(tope_pisos)
pisos_gem = F.least(F.col("pisos"), tope_pisos)
im_valor = (F.when(F.col("im") == "PGA", F.col("PGA_g")).when(F.col("im") == "SA(0.3s)", F.col("Sa03_g"))
             .when(F.col("im") == "SA(0.6s)", F.col("Sa06_g")).otherwise(F.col("Sa10_g")))
dano = (spark.table("riesgo_demanda").join(censo, "ubigeo", "left").fillna(0.0, subset=["p_adobe", "p_quincha", "p_madera"])
        .withColumn("tipologia", tipologia).withColumn("pisos_gem", pisos_gem)
        .join(F.broadcast(spark.table("riesgo_fragilidad_parametros").drop("fuente", "taxonomia_gem")), ["tipologia", "pisos_gem"])
        .withColumn("IM_g", im_valor))

T_IM = {"PGA": float(PERIODOS[0]), "SA(0.3s)": 0.3, "SA(0.6s)": 0.6, "SA(1.0s)": 1.0}


# Interpola una malla de amplificación en un periodo fijo.
def en_periodo(grilla, T):
    j = int(np.clip(np.searchsorted(PERIODOS, T) - 1, 0, len(PERIODOS) - 2))
    wj = float(np.clip((np.log(T) - np.log(PERIODOS[j])) / (np.log(PERIODOS[j + 1]) - np.log(PERIODOS[j])), 0, 1))
    return np.exp((1 - wj) * np.log(grilla[:, j]) + wj * np.log(grilla[:, j + 1]))


G = {c: amp[c].to_numpy().reshape(len(VS30_GRID), len(PERIODOS)) for c in ("F", "F_p10", "F_p90")}
razones = pd.concat([pd.DataFrame({"vs30_bin": VS30_GRID, "im": im,
                                   "r_p10": en_periodo(G["F_p10"], T) / en_periodo(G["F"], T),
                                   "r_p90": en_periodo(G["F_p90"], T) / en_periodo(G["F"], T)})
                     for im, T in T_IM.items()], ignore_index=True)
dano = (dano
        .withColumn("vs30_bin", F.least(F.lit(float(VS30_GRID.max())),
                                        F.greatest(F.lit(float(VS30_GRID.min())), F.round(F.col("vs30") / 10) * 10)))
        .join(F.broadcast(a_spark(razones, "vs30_bin double, im string, r_p10 double, r_p90 double")), ["vs30_bin", "im"], "left")
        .withColumn("r_p10", F.when(F.col("tipo") == "norma", F.lit(1.0)).otherwise(F.coalesce("r_p10", F.lit(1.0))))
        .withColumn("r_p90", F.when(F.col("tipo") == "norma", F.lit(1.0)).otherwise(F.coalesce("r_p90", F.lit(1.0)))))
SIGMA_DEMANDA = {"registrado": SIGMA_LOSO, "diseno": SIGMA_LOSO, "0.25g": SIGMA_LOSO, "0.70g": SIGMA_LOSO, "norma_sitio": SIGMA_F_SITIO, "norma": 0.0}
print("σ de la demanda por tipo de escenario (ln):", {k: round(v, 3) for k, v in SIGMA_DEMANDA.items()})
sigma_dem = F.lit(0.0)
for tipo_esc, s in SIGMA_DEMANDA.items():
    sigma_dem = F.when(F.col("tipo") == tipo_esc, F.lit(float(s))).otherwise(sigma_dem)
dano = (dano.withColumn("sigma_demanda", sigma_dem)
        .withColumn("beta_total", F.sqrt(F.col("beta") ** 2 + F.col("sigma_demanda") ** 2)))
for e in ESTADOS:
    dano = dano.withColumn(f"P_ge_{e}", phi(F.log(F.col("IM_g") / F.col(f"theta_{e}")) / F.col("beta_total")))
dano = (dano
        .withColumn("P_sin_dano", 1 - F.col("P_ge_leve"))
        .withColumn("P_leve", F.col("P_ge_leve") - F.col("P_ge_moderado"))
        .withColumn("P_moderado", F.col("P_ge_moderado") - F.col("P_ge_severo"))
        .withColumn("P_severo", F.col("P_ge_severo") - F.col("P_ge_colapso"))
        .withColumn("P_colapso", F.col("P_ge_colapso"))
        .withColumn("P_colapso_bajo", phi(F.log(F.col("IM_g") * F.col("r_p10") / F.col("theta_colapso")) / F.col("beta_total")))
        .withColumn("P_colapso_alto", phi(F.log(F.col("IM_g") * F.col("r_p90") / F.col("theta_colapso")) / F.col("beta_total")))
        .withColumn("dano_esperado", sum(F.col(f"P_ge_{e}") for e in ESTADOS))
        .withColumn("ln_Sa_Tb", F.log("Sa_Tb")))

In [ ]:
# Guarda el daño de cada edificio por escenario.
COLS_DANO = ["id", "escenario", "evento", "tipo", "ubigeo", "distrito", "provincia", "h3_id", "h3_r8", "tipologia", "pisos", "Tb", "Tb_TP",
             "vs30", "tipo_suelo", "Sa_Tb", "ln_Sa_Tb", "pgv_cms", "PGA_g", "Sa03_g", "Sa06_g", "Sa10_g", "im", "IM_g", "theta_colapso",
             "sigma_demanda", "P_sin_dano", "P_leve", "P_moderado", "P_severo", "P_colapso", "P_colapso_bajo", "P_colapso_alto",
             "dano_esperado", "fuera_cobertura"]
escribir(dano.select(*COLS_DANO), "riesgo_dano_edificios", particion="escenario")
spark.table("riesgo_dano_edificios").groupBy("escenario").agg(F.count("*").alias("edificios"), F.round(F.avg("P_colapso"), 4).alias("P_colapso_media"),
                                                              F.round(F.avg("dano_esperado"), 3).alias("dano_esperado_medio")).orderBy("escenario").show(truncate=False)

## 10. Feature engineering del modelo de daño (Tainan y Manta)
La sacudida en cada edificio de Tainan y Manta se calcula con la misma cadena que en Lima (roca krigeada + amplificación + suelo no lineal), así el modelo usa las mismas variables en las tres ciudades.

In [ ]:
EPSG_CIUDAD = {"Tainan": 32651, "Manta": 32717}
ESQ_FE_DANO = ("ciudad string, id string, lat double, lon double, dano string, severo boolean, pisos long, altura_m double, area_m2 double, Tb double, vs30 double, "
               "suelo_cod double, PGA_g double, Sa_Tb double, Sa03_g double, Sa06_g double, Sa10_g double, PGV_cms double, Tp_s double, D595_s double, Arias_ms double, "
               "etiqueta double")


# Logaritmo de la amplificación con el suelo no lineal de BSSA14.
def ln_F(periodo, vs30, pga_r):
    periodo, vs30, pga_r = np.broadcast_arrays(np.asarray(periodo, float), np.asarray(vs30, float), np.asarray(pga_r, float))
    f4 = np.interp(np.log(periodo), np.log(PERIODOS), F4_NL)
    f5 = np.interp(np.log(periodo), np.log(PERIODOS), F5_NL)
    return ln_amplificacion(periodo, vs30) + f4 * (np.exp(f5 * (np.minimum(vs30, 760.0) - 360.0)) - np.exp(f5 * 400.0)) * (np.log((pga_r + 0.1) / 0.1) - np.log((PGA_REF_NL + 0.1) / 0.1))


# PGA en roca por punto fijo, considerando el suelo no lineal.
def pga_en_roca(pga, vs30):
    r = pga
    for _ in range(8):
        r = pga / np.exp(ln_F(PERIODOS[0], vs30, r))
    return r


# Interpola cada fila de un espectro en su propio periodo.
def en_periodo_fila(ln_sa, periodo):
    return np.array([np.interp(np.log(periodo[j]), np.log(PERIODOS), ln_sa[j]) for j in range(len(ln_sa))])


# Aplica la cadena de demanda a los edificios de una ciudad.
def variables_ciudad(edif_c, est_c, epsg):
    est_c = est_c.reset_index(drop=True)
    pga_r = np.array([float(pga_en_roca(p, v)) for p, v in zip(est_c.pga_geo_g, est_c.vs30)])
    ln_sa_roca = np.log(np.array(est_c.sa_geo.tolist(), float)) - np.array([ln_F(PERIODOS, v, r) for v, r in zip(est_c.vs30, pga_r)])
    extras = {"PGV_cms": "pgv_geo_cms", "Tp_s": "tp_s", "D595_s": "d595_s", "Arias_ms": "arias_ms"}
    if len(est_c) == 1:
        pesos = np.ones((len(edif_c), 1))
    else:
        transf = Transformer.from_crs(4326, epsg, always_xy=True).transform
        xy_e = np.column_stack(transf(est_c.lon.to_numpy(), est_c.lat.to_numpy())) / 1000
        xy_b = np.column_stack(transf(edif_c.lon.to_numpy(), edif_c.lat.to_numpy())) / 1000
        pep, rango_v = ajustar_variograma(xy_e, ((ln_sa_roca - ln_sa_roca.mean(0)) / ln_sa_roca.std(0).clip(1e-6)).T)
        pesos = pesos_kriging(xy_e, xy_b, pep, rango_v)
    salida = edif_c.copy()
    pga_b = np.exp(pesos @ np.log(pga_r))
    sa_b = pesos @ ln_sa_roca
    vs_b, tb_b = salida.vs30.to_numpy(float), salida.Tb.to_numpy(float)
    salida["PGA_g"] = pga_b * np.exp(ln_F(PERIODOS[0], vs_b, pga_b))
    salida["Sa_Tb"] = np.exp(en_periodo_fila(sa_b, tb_b) + ln_F(tb_b, vs_b, pga_b))
    for col, periodo in (("Sa03_g", 0.3), ("Sa06_g", 0.6), ("Sa10_g", 1.0)):
        salida[col] = np.exp(en_periodo_fila(sa_b, np.full(len(salida), periodo)) + ln_F(np.full(len(salida), periodo), vs_b, pga_b))
    for col, origen in extras.items():
        salida[col] = np.exp(pesos @ np.log(est_c[origen].to_numpy(float)))
    return salida


# Se ejecuta por ciudad: variables del modelo de daño y etiqueta.
def variables_grupo(edif_c, est_c):
    if edif_c.empty or est_c.empty:
        return pd.DataFrame(columns=[c.split()[0] for c in ESQ_FE_DANO.split(", ")])
    salida = variables_ciudad(edif_c, est_c, EPSG_CIUDAD[edif_c.ciudad.iloc[0]])
    salida["etiqueta"] = salida.severo.astype(float)
    return salida


edif_dano = spark.table("gold_edificios_dano")
reg_dano = spark.table("gold_registros").where("ciudad <> 'Lima'").select("ciudad", "lat", "lon", "vs30", "pga_geo_g", "pgv_geo_cms", "sa_geo", "tp_s", "d595_s", "arias_ms")
escribir(edif_dano.groupBy("ciudad").cogroup(reg_dano.groupBy("ciudad")).applyInPandas(variables_grupo, ESQ_FE_DANO), "fe_dano")
display(spark.table("fe_dano").groupBy("ciudad").agg(*[F.round(F.median(c), 3).alias(c) for c in ["PGA_g", "PGV_cms", "Sa03_g", "Sa06_g", "Sa10_g", "Sa_Tb"]]))

In [ ]:
# Variables del modelo de daño.
VARS_EDIFICIO = ["pisos", "Tb", "area_m2"]
VARS_SUELO = ["vs30", "suelo_cod"]
VARS_ONDA = ["PGA_g", "PGV_cms", "Sa03_g", "Sa06_g", "Sa10_g", "Sa_Tb"]
VARS_DANO = VARS_EDIFICIO + VARS_SUELO + VARS_ONDA
filas_sel = [("dano", c, "num", "tainan_manta") for c in VARS_DANO]

## 11. Sacudida en los edificios de Turquía (ShakeMap)
Para validar fuera de Tainan y Manta se usa el sismo de Kahramanmaraş 2023. La sacudida en cada edificio sale del ShakeMap del USGS (interpolación bilineal de la grilla) y no de la cadena propia, porque el pipeline no tiene registros de esas ciudades. Sa(Tb) se interpola en log-log entre PGA (T = 0.05 s) y PSA a 0.3, 0.6, 1.0 y 3.0 s. Las variables quedan en `fe_dano_turquia`, con las mismas columnas de `fe_dano` (Tp, D5-95 y Arias quedan nulos), más la desviación estándar del ShakeMap en cada punto.

In [ ]:
# Sacudida del ShakeMap en cada edificio de Turquía, con las mismas columnas de fe_dano.
hay_turquia = spark.catalog.tableExists("gold_turquia_edificios") and spark.catalog.tableExists("gold_turquia_shakemap")
COLS_SM_TR = ["mmi", "pga_g", "pgv_cms", "sa03_g", "sa06_g", "sa10_g", "sa30_g", "vs30_shakemap", "sd_ln_pga", "sd_ln_sa03", "sd_ln_sa06", "sd_ln_sa10"]
PERIODOS_SM = np.array([PERIODOS[0], 0.3, 0.6, 1.0, 3.0])
ESQ_FE_DANO_TR = (ESQ_FE_DANO + ", fuente_sacudida string, mmi double, vs30_shakemap double, sd_ln_pga double, sd_ln_sa03 double, "
                  "sd_ln_sa06 double, sd_ln_sa10 double, altura_gba_m double, altura_imputada boolean")
COLS_FE_DANO_TR = [c.split()[0] for c in ESQ_FE_DANO_TR.split(", ")]


# Se ejecuta por ciudad: interpola el ShakeMap en cada edificio y Sa(Tb) en log-log.
def sacudida_shakemap(edif_c, sm_c):
    if edif_c.empty or sm_c.empty:
        return pd.DataFrame(columns=COLS_FE_DANO_TR)
    lons, lats = np.sort(sm_c.lon.unique()), np.sort(sm_c.lat.unique())
    puntos = np.column_stack([edif_c.lat.to_numpy(float), edif_c.lon.to_numpy(float)])
    v = {}
    for c in COLS_SM_TR:
        malla = sm_c.pivot_table(index="lat", columns="lon", values=c, dropna=False).reindex(index=lats, columns=lons).to_numpy(float)
        v[c] = RegularGridInterpolator((lats, lons), malla, bounds_error=False, fill_value=None)(puntos)
    salida = edif_c.copy()
    ln_sa = np.log(np.column_stack([v["pga_g"], v["sa03_g"], v["sa06_g"], v["sa10_g"], v["sa30_g"]]))
    tb = np.clip(salida.Tb.to_numpy(float), PERIODOS_SM[0], PERIODOS_SM[-1])
    salida["Sa_Tb"] = np.exp([np.interp(np.log(t), np.log(PERIODOS_SM), fila) for t, fila in zip(tb, ln_sa)])
    for col, origen in (("PGA_g", "pga_g"), ("PGV_cms", "pgv_cms"), ("Sa03_g", "sa03_g"), ("Sa06_g", "sa06_g"), ("Sa10_g", "sa10_g")):
        salida[col] = v[origen]
    for c in ("mmi", "vs30_shakemap", "sd_ln_pga", "sd_ln_sa03", "sd_ln_sa06", "sd_ln_sa10"):
        salida[c] = v[c]
    for c in ("Tp_s", "D595_s", "Arias_ms"):
        salida[c] = np.nan
    salida["etiqueta"] = salida.severo.astype(float)
    salida["fuente_sacudida"] = "shakemap_usgs"
    return salida[COLS_FE_DANO_TR]


if hay_turquia:
    edif_tr = spark.table("gold_turquia_edificios").select("ciudad", "id", "lat", "lon", "dano", "severo", "pisos", "altura_m", "area_m2", "Tb", "vs30",
                                                           "suelo_cod", "altura_gba_m", "altura_imputada")
    sm_tr = spark.table("gold_turquia_shakemap").select("ciudad", "lon", "lat", *COLS_SM_TR)
    escribir(edif_tr.groupBy("ciudad").cogroup(sm_tr.groupBy("ciudad")).applyInPandas(sacudida_shakemap, ESQ_FE_DANO_TR), "fe_dano_turquia")
    display(spark.table("fe_dano_turquia").groupBy("ciudad")
            .agg(F.count("*").alias("edificios"), F.round(F.avg("etiqueta"), 3).alias("tasa_danado"),
                 *[F.round(F.median(c), 3).alias(c) for c in ["PGA_g", "PGV_cms", "Sa03_g", "Sa06_g", "Sa10_g", "Sa_Tb"]]).orderBy("ciudad"))
else:
    print("Sin gold_turquia_edificios: no se valida en Turquía (ver 01_bronze)")

## 12. Validación de la fragilidad con el daño observado en Turquía
¿La cadena intensidad → fragilidad → daño predice el daño que ocurrió? Se compara el daño visible en imágenes (Liu, 2026) con P(daño ≥ severo) de las curvas GEM. En el caso base, los edificios de 1 o 2 pisos son de albañilería de ladrillo no reforzada (`MUR-CL99/LWAL-DNO`) y los de 3 o más son de concreto con tabiquería de ductilidad baja (`CR/LFINF-DUL`). Cada edificio usa la curva de su altura; por encima de la mayor publicada se usa la última. A β se le suma la incertidumbre del ShakeMap en el punto: √(β² + σ²). La sensibilidad cambia la tipología y el estado de daño.

Métricas por ciudad: tasa observada frente a probabilidad media, Brier, AUC y F1 con umbral 0.5. La métrica global usa las dos ciudades turcas. Jindires (Siria) queda fuera del dominio, porque el ShakeMap no tiene estaciones allí. La tabla por cuartil de Vs30 de Antakya muestra si la grilla del ShakeMap resuelve la variación dentro de la ciudad.

El ShakeMap tiene una resolución cercana a 1 km, así que no puede distinguir edificios vecinos. Por eso también se compara por zonas de unos 500 m con al menos 20 edificios: la tasa observada de cada zona frente a su probabilidad media (`riesgo_validacion_turquia_zonas`), con la correlación de Spearman y el sesgo por ciudad.

In [ ]:
# P(daño ≥ estado) con las curvas GEM para cada edificio y tipología, y métricas por ciudad.
TIPOLOGIAS_TR = {"mixta_MUR_CR-DUL": ("albanileria_no_reforzada", "concreto_armado_dul"),
                 "mixta_MUR_CR-DUM": ("albanileria_no_reforzada", "concreto_armado"),
                 "solo_CR-DUL": ("concreto_armado_dul", "concreto_armado_dul"),
                 "solo_CR-DUM": ("concreto_armado", "concreto_armado")}
TIPOLOGIA_BASE, ESTADO_BASE, PISOS_CORTE_TR = "mixta_MUR_CR-DUL", "severo", 2
CIUDADES_DOMINIO_TR = ["Antakya", "Nurdagi"]
ESQ_VAL_TR = ("tipologia_caso string, ciudad string, estado string, edificios long, tasa_observada double, prob_media_predicha double, "
              "error_tasa double, brier double, auc double, f1_umbral_05 double, recall_umbral_05 double")


# Se ejecuta por caso, ciudad y estado: calibración, Brier, AUC por rangos y F1 con umbral 0.5.
def metricas_validacion(g):
    y, p = g.etiqueta.to_numpy(float), g.P.to_numpy(float)
    n_pos, auc = y.sum(), np.nan
    if 0 < n_pos < len(y) and np.ptp(p) > 0:
        r = pd.Series(p).rank().to_numpy()
        auc = (r[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * (len(y) - n_pos))
    pred = p >= 0.5
    tp, fp, fn = (pred & (y == 1)).sum(), (pred & (y == 0)).sum(), (~pred & (y == 1)).sum()
    return pd.DataFrame([{"tipologia_caso": g.tipologia_caso.iloc[0], "ciudad": g.ciudad.iloc[0], "estado": g.estado.iloc[0], "edificios": len(y),
                          "tasa_observada": y.mean(), "prob_media_predicha": p.mean(), "error_tasa": p.mean() - y.mean(),
                          "brier": np.mean((p - y) ** 2), "auc": auc, "f1_umbral_05": 2 * tp / max(2 * tp + fp + fn, 1),
                          "recall_umbral_05": tp / max(tp + fn, 1)}])


if hay_turquia:
    frag_tr = spark.table("riesgo_fragilidad_parametros").drop("fuente")
    tope_tr = frag_tr.groupBy("tipologia").agg(F.max("pisos_gem").alias("tope"))
    casos_tr = spark.createDataFrame([(k, b, a) for k, (b, a) in TIPOLOGIAS_TR.items()], "tipologia_caso string, bajo string, alto string")
    sd_im = (F.when(F.col("im") == "PGA", F.col("sd_ln_pga")).when(F.col("im") == "SA(0.3s)", F.col("sd_ln_sa03"))
             .when(F.col("im") == "SA(0.6s)", F.col("sd_ln_sa06")).otherwise(F.col("sd_ln_sa10")))
    p_tr = (spark.table("fe_dano_turquia").crossJoin(F.broadcast(casos_tr))
            .withColumn("tipologia", F.when(F.col("pisos") <= PISOS_CORTE_TR, F.col("bajo")).otherwise(F.col("alto")))
            .join(F.broadcast(tope_tr), "tipologia").withColumn("pisos_gem", F.least("pisos", "tope"))
            .join(F.broadcast(frag_tr), ["tipologia", "pisos_gem"])
            .withColumn("IM_g", im_valor)
            .withColumn("beta_total", F.sqrt(F.col("beta") ** 2 + F.coalesce(sd_im, F.lit(0.0)) ** 2)))
    for e in ESTADOS:
        p_tr = p_tr.withColumn(f"P_ge_{e}", phi(F.log(F.col("IM_g") / F.col(f"theta_{e}")) / F.col("beta_total")))
    escribir(p_tr.where(F.col("tipologia_caso") == TIPOLOGIA_BASE)
             .select("ciudad", "id", "lat", "lon", "pisos", "vs30", "vs30_shakemap", "tipologia", "taxonomia_gem", "pisos_gem", "im", "IM_g", "beta_total",
                     "etiqueta", *[f"P_ge_{e}" for e in ESTADOS]), "riesgo_validacion_turquia_edificios")
    largo = (p_tr.select("tipologia_caso", "ciudad", "etiqueta",
                         F.explode(F.array(*[F.struct(F.lit(e).alias("estado"), F.col(f"P_ge_{e}").alias("P")) for e in ("moderado", "severo", "colapso")])).alias("x"))
             .select("tipologia_caso", "ciudad", "etiqueta", "x.*"))
    validacion = (largo.groupBy("tipologia_caso", "ciudad", "estado").applyInPandas(metricas_validacion, ESQ_VAL_TR)
                  .join(F.broadcast(casos_tr.withColumnRenamed("bajo", "tipologia_bajos").withColumnRenamed("alto", "tipologia_altos")), "tipologia_caso"))
    escribir(validacion, "riesgo_validacion_turquia")

In [ ]:
# Caso base por ciudad, sensibilidad, métricas registradas y diagnóstico por cuartil de Vs30 en Antakya.
if hay_turquia:
    val = spark.table("riesgo_validacion_turquia").toPandas()
    base = val[(val.tipologia_caso == TIPOLOGIA_BASE) & (val.estado == ESTADO_BASE)].sort_values("ciudad")
    print(f"Caso base: {TIPOLOGIA_BASE} ({' / '.join(TIPOLOGIAS_TR[TIPOLOGIA_BASE])}), daño observado frente a P(≥ {ESTADO_BASE})")
    display(base[["ciudad", "edificios", "tasa_observada", "prob_media_predicha", "error_tasa", "brier", "auc", "f1_umbral_05"]].round(3))
    print("Sensibilidad: error de la tasa por tipología y estado de daño")
    display(val.pivot_table(index=["tipologia_caso", "estado"], columns="ciudad", values="error_tasa").round(3).reset_index())
    for r in base.itertuples():
        registrar_modelo(f"validacion_turquia_{r.ciudad.lower()}", {"tipologia": TIPOLOGIA_BASE, "comparado_con": f"P(>= {ESTADO_BASE})",
                                                                    "fuente_obs": "zenodo_18437501 (interpretacion visual)"},
                         {"tasa_observada": r.tasa_observada, "prob_media_predicha": r.prob_media_predicha, "error_tasa": r.error_tasa,
                          "brier": r.brier, "auc": r.auc})
    dominio = base[base.ciudad.isin(CIUDADES_DOMINIO_TR)]
    if len(dominio):
        registrar_modelo("validacion_turquia_global", {"tipologia": TIPOLOGIA_BASE, "ciudades": ", ".join(CIUDADES_DOMINIO_TR)},
                         {"error_abs_tasa_medio": dominio.error_tasa.abs().mean(), "brier_ponderado": float(np.average(dominio.brier, weights=dominio.edificios))})

    antakya_tr = spark.table("fe_dano_turquia").where(F.col("ciudad") == "Antakya").withColumn("cuartil_vs30", F.ntile(4).over(Window.orderBy("vs30_shakemap")))
    display(antakya_tr.groupBy("cuartil_vs30").agg(F.count("*").alias("edificios"), F.round(F.median("vs30_shakemap"), 0).alias("vs30_shakemap"),
                                                   F.round(F.median("PGA_g"), 3).alias("pga_shakemap_g"), F.round(F.avg("etiqueta"), 3).alias("tasa_danado_obs"),
                                                   F.round(F.avg("pisos"), 2).alias("pisos_medio")).orderBy("cuartil_vs30"))

    b = val[val.tipologia_caso == TIPOLOGIA_BASE].pivot_table(index="ciudad", columns="estado", values="prob_media_predicha")[["moderado", "severo", "colapso"]]
    observado = val.groupby("ciudad").tasa_observada.first().loc[b.index]
    x = np.arange(len(b))
    fig, ax = plt.subplots(figsize=(8, 4))
    for k, e in enumerate(b.columns):
        ax.bar(x + (k - 1.5) * 0.2, b[e], width=0.2, label=f"P(≥ {e})")
    ax.bar(x + 0.3, observado, width=0.2, color="k", label="observado")
    ax.set_xticks(x, b.index)
    ax.set(ylabel="Fracción de edificios", title=f"Daño en Turquía 2023: GEM {TIPOLOGIA_BASE} frente al observado")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=.3)
    plt.tight_layout(); plt.show()

In [ ]:
# Validación por zonas de ~500 m: el ShakeMap tiene ~1 km de resolución y no distingue edificios vecinos.
TAM_ZONA_GRADOS, MIN_EDIF_ZONA = 0.005, 20
if hay_turquia:
    zonas_tr = (spark.table("riesgo_validacion_turquia_edificios")
                .withColumn("zona", F.concat_ws("_", F.floor(F.col("lon") / TAM_ZONA_GRADOS), F.floor(F.col("lat") / TAM_ZONA_GRADOS)))
                .groupBy("ciudad", "zona")
                .agg(F.count("*").alias("edificios"), F.avg("lon").alias("lon"), F.avg("lat").alias("lat"), F.avg("pisos").alias("pisos_medio"),
                     F.avg("vs30_shakemap").alias("vs30_shakemap"), F.avg("IM_g").alias("im_medio_g"),
                     F.avg("etiqueta").alias("tasa_observada"), F.avg(f"P_ge_{ESTADO_BASE}").alias("prob_media_predicha"))
                .where(F.col("edificios") >= MIN_EDIF_ZONA))
    escribir(zonas_tr, "riesgo_validacion_turquia_zonas")
    z = spark.table("riesgo_validacion_turquia_zonas").toPandas()
    filas_z = []
    for ciudad, g in z.groupby("ciudad"):
        dif = g.prob_media_predicha - g.tasa_observada
        filas_z.append({"ciudad": ciudad, "zonas": len(g), "edificios": int(g.edificios.sum()),
                        "spearman": g.tasa_observada.rank().corr(g.prob_media_predicha.rank()) if len(g) > 2 else np.nan,
                        "sesgo": np.average(dif, weights=g.edificios), "error_abs": np.average(dif.abs(), weights=g.edificios)})
    resumen_z = pd.DataFrame(filas_z)
    print(f"Zonas de {TAM_ZONA_GRADOS}° con al menos {MIN_EDIF_ZONA} edificios: tasa observada frente a P(≥ {ESTADO_BASE}) media")
    display(resumen_z.round(3))
    for r in resumen_z.itertuples():
        registrar_modelo(f"validacion_turquia_zonas_{r.ciudad.lower()}", {"tipologia": TIPOLOGIA_BASE, "zona_grados": TAM_ZONA_GRADOS, "min_edificios": MIN_EDIF_ZONA},
                         {"zonas": r.zonas, "spearman": r.spearman, "sesgo": r.sesgo, "error_abs": r.error_abs})

    fig, ax = plt.subplots(figsize=(6, 5))
    for ciudad, g in z.groupby("ciudad"):
        ax.scatter(g.prob_media_predicha, g.tasa_observada, s=10 + 150 * g.edificios / z.edificios.max(), alpha=.6, label=ciudad)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set(xlim=(0, 1), ylim=(0, 1.02), xlabel=f"P(≥ {ESTADO_BASE}) media de la zona (GEM {TIPOLOGIA_BASE})", ylabel="Fracción dañada observada",
           title=f"Turquía 2023 por zonas de {TAM_ZONA_GRADOS}° (~{TAM_ZONA_GRADOS * 111:.1f} km)")
    ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 13. Feature engineering de los perfiles de riesgo (Lima)
Un edificio por fila bajo el escenario de la Norma E.030 con amplificación del suelo. Se descartan las variables con |correlación| > 0.90.

In [ ]:
# Variables de los perfiles de riesgo: descarta las muy correlacionadas.
ESCENARIO_PERFILES = "e030_sitio"
lote_clustering = f"amp_v{VERSION_AMP}"
base_km = (spark.table("riesgo_dano_edificios").where(F.col("escenario") == ESCENARIO_PERFILES)
           .select(F.lit(lote_clustering).alias("lote"), "id",
                   F.log(F.col("IM_g") / F.col("theta_colapso")).alias("ln_IM_theta"),
                   F.col("ln_Sa_Tb").cast("double").alias("ln_Sa_Tb"),
                   F.col("Tb_TP").cast("double").alias("Tb_TP"),
                   F.col("vs30").cast("double").alias("vs30"),
                   F.log(F.col("pisos").cast("double")).alias("ln_pisos"),
                   "Sa_Tb", "IM_g", "theta_colapso", "pisos", "P_colapso", "dano_esperado", "tipologia", "distrito"))

cols_id = ["lote", "id"]
cols_resultado = ["P_colapso", "dano_esperado", "tipologia", "distrito"]
cols_redundantes = ["Sa_Tb", "IM_g", "theta_colapso", "pisos"]
cols_excluir = cols_id + cols_resultado + cols_redundantes
cols_nums = [c.name for c in base_km.schema if c.dataType != StringType() and c.name not in cols_excluir]
print("Candidatas:", cols_nums)

corr_mat = Correlation.corr(VectorAssembler(inputCols=cols_nums, outputCol="num_vec").transform(base_km).select("num_vec"),
                            "num_vec", "pearson").head()[0].toArray()
to_drop = set()
for i in range(len(cols_nums)):
    for j in range(i + 1, len(cols_nums)):
        if abs(corr_mat[i, j]) > 0.90:
            to_drop.add(cols_nums[j])
num_sel = [c for c in cols_nums if c not in to_drop]
print("Drop por multicolinealidad:", sorted(to_drop), "| variables:", num_sel)
display(pd.DataFrame(corr_mat, index=cols_nums, columns=cols_nums).round(2))

if spark.catalog.tableExists("fe_clustering"):
    base_km.write.mode("overwrite").option("replaceWhere", f"lote = '{lote_clustering}'").saveAsTable("fe_clustering")
else:
    base_km.write.partitionBy("lote").saveAsTable("fe_clustering")
filas_sel += [("clustering", c, "num", lote_clustering) for c in num_sel]
print("fe_clustering:", spark.table("fe_clustering").where(F.col("lote") == lote_clustering).count(), "filas")

## 14. Acelerograma sintético para la simulación en tiempo real
El modelo estocástico de Boore se calibra con los registros de Lima llevados a roca con el champion de amplificación: se elige Δσ, κ y Q0 en una grilla y una corrección por periodo, con validación dejando fuera un sismo. Los parámetros se guardan en `riesgo_parametros`; `11` los lee y no ajusta nada.

In [ ]:
# Espectro de respuesta y modelo estocástico de Boore; 11 genera sus señales con las mismas funciones.
RHO_G_CM3, BETA_KM_S = 2.8, 3.7
FS_SIM = 100
PERIODOS_CAL = PERIODOS[[int(np.abs(PERIODOS - t).argmin()) for t in (0.05, 0.1, 0.2, 0.3, 0.6, 1.0, 2.0)]]
GRILLA_SINT = [(ds, ka, q0) for ds in (100.0, 300.0, 1000.0, 3000.0) for ka in (0.005, 0.02) for q0 in (300.0, 680.0, 1500.0)]
ESQ_TAREA_SINT = "clave string, j long, m double, r double, semilla long, ds double, ka double, q0 double, correccion array<double>"


_CACHE_SDOF = {}


# Filtro discreto de un oscilador de un grado de libertad; guarda en caché cada periodo.
def _filtro_sdof(periodo, dt, z):
    if (periodo, dt, z) in _CACHE_SDOF:
        return _CACHE_SDOF[(periodo, dt, z)]
    w = 2 * np.pi / periodo
    A = np.array([[0.0, 1.0], [-w * w, -2 * z * w]])
    B = np.array([[0.0], [-1.0]])
    C = np.array([[1.0, 0.0]])
    D = np.array([[0.0]])
    Ad, Bd, Cd, Dd, _ = cont2discrete((A, B, C, D), dt, method="foh")
    b, a = ss2tf(Ad, Bd, Cd, Dd)
    _CACHE_SDOF[(periodo, dt, z)] = (b[0], a, w)
    return _CACHE_SDOF[(periodo, dt, z)]


# Espectro de respuesta Sa con 5 % de amortiguamiento.
def espectro_sa(acc_g, dt, periodos=PERIODOS, z=AMORTIGUAMIENTO):
    sa = np.empty(len(periodos))
    for k, periodo in enumerate(periodos):
        b, a, w = _filtro_sdof(float(periodo), float(dt), z)
        sa[k] = w * w * np.abs(lfilter(b, a, acc_g)).max()
    return sa


# Espectro de Fourier del modelo estocástico: fuente, distancia y atenuación.
def espectro_fourier(fr, m, r, ds, ka, q0):
    m0 = 10 ** (1.5 * m + 16.05)
    fc = 4.9e6 * BETA_KM_S * (ds / m0) ** (1 / 3)
    re_km = np.sqrt(r ** 2 + (10 ** (-0.405 + 0.235 * m)) ** 2)
    g = 1 / re_km if re_km <= 70 else (1 / 70 if re_km <= 130 else np.sqrt(130 / re_km) / 70)
    q = np.maximum(q0 * fr ** 0.5, 50.0)
    c = 0.55 * 0.707 * 2.0 / (4 * np.pi * RHO_G_CM3 * BETA_KM_S ** 3) * 1e-20
    return c * m0 * (2 * np.pi * fr) ** 2 / (1 + (fr / fc) ** 2) * g * np.exp(-np.pi * fr * re_km / (q * BETA_KM_S)) * np.exp(-np.pi * ka * fr), fc, re_km


# Acelerograma estocástico en roca.
def sintetico_roca(m, r, semilla, ds, ka, q0, correccion=None):
    rng = np.random.default_rng(semilla)
    _, fc, re_km = espectro_fourier(np.array([1.0]), m, r, ds, ka, q0)
    tgm = 1 / fc + 0.05 * re_km
    n = int(2 ** np.ceil(np.log2((2.5 * tgm + 20) * FS_SIM)))
    t = np.arange(n) / FS_SIM
    eps, tn = 0.2, 2 * tgm
    b = -eps * np.log(0.05) / (1 + eps * (np.log(eps) - 1))
    ventana = (np.e / (eps * tn)) ** b * np.maximum(t, 1e-9) ** b * np.exp(-(b / (eps * tn)) * t)
    x = np.fft.rfft(rng.standard_normal(n) * ventana)
    x /= np.sqrt(np.mean(np.abs(x) ** 2))
    fr = np.fft.rfftfreq(n, 1 / FS_SIM)
    fr[0] = fr[1]
    a, _, _ = espectro_fourier(fr, m, r, ds, ka, q0)
    if correccion is not None:
        a = a * np.exp(np.interp(np.log(1 / fr), np.log(PERIODOS_CAL), correccion))
    return np.fft.irfft(a * x * FS_SIM, n)


# Espectro medio en logaritmo de las dos componentes sintéticas.
def ln_sa_sintetico(m, r, semilla, params, correccion=None):
    e = sintetico_roca(m, r, semilla, *params, correccion) / G_CM_S2
    n = sintetico_roca(m, r, semilla + 1, *params, correccion) / G_CM_S2
    return 0.5 * (np.log(espectro_sa(e, 1 / FS_SIM, PERIODOS_CAL)) + np.log(espectro_sa(n, 1 / FS_SIM, PERIODOS_CAL)))


# Se ejecuta en los workers: calcula el espectro de cada sismo sintético.
def calcular_sinteticos(lotes):
    for pdf in lotes:
        filas = []
        for t in pdf.itertuples():
            correccion = None if t.correccion is None else np.asarray(t.correccion, float)
            filas.append({"clave": t.clave, "j": t.j, "ln_sa": ln_sa_sintetico(t.m, t.r, t.semilla, (t.ds, t.ka, t.q0), correccion).tolist()})
        yield pd.DataFrame(filas)


# Reparte las tareas de sismos sintéticos entre los workers y junta los espectros.
def correr_sinteticos(tareas):
    salida = (spark.createDataFrame(tareas, ESQ_TAREA_SINT).repartition(16)
              .mapInPandas(calcular_sinteticos, "clave string, j long, ln_sa array<double>").toPandas())
    return {(f.clave, int(f.j)): np.asarray(f.ln_sa, float) for f in salida.itertuples()}

In [ ]:
# Calibra Δσ, κ, Q0 y una corrección por periodo con los registros de Lima en roca; valida dejando fuera un sismo.
obs_sint = REG_LIMA.where("vs30 IS NOT NULL AND dist_hipo_km IS NOT NULL").orderBy("evento", "archivo_estacion").select("evento", "vs30", "dist_hipo_km", "sa_geo").toPandas()
obs_sint["m"] = obs_sint.evento.str.extract(r"_M[wl]?(\d+\.\d+)$")[0].astype(float)
obs_sint = pd.concat([g.sample(min(8, len(g)), random_state=SEED) for _, g in obs_sint.groupby("evento")], ignore_index=True)
sa_obs = np.array(obs_sint.sa_geo.tolist(), float)
y_sint = np.array([np.interp(np.log(PERIODOS_CAL), np.log(PERIODOS), np.log(sa_obs[j])) - ln_amplificacion(PERIODOS_CAL, obs_sint.vs30[j]) for j in range(len(obs_sint))])
sint_grilla = correr_sinteticos([(str(g), int(f.Index), float(f.m), float(f.dist_hipo_km), int(7 + f.Index), *g, None) for g in GRILLA_SINT for f in obs_sint.itertuples()])
errores_sint = {g: np.array([sint_grilla[(str(g), int(j))] for j in obs_sint.index]) - y_sint for g in GRILLA_SINT}


# Elige los parámetros de la grilla y la corrección por periodo que minimizan el error.
def ajustar_sintetico(mascara):
    pesos_ev = 1 / obs_sint[mascara].groupby("evento").evento.transform("count").to_numpy()

    # Error de un punto de la grilla, con el mismo peso para cada sismo.
    def coste(g):
        c = -np.average(errores_sint[g][mascara], axis=0, weights=pesos_ev)
        return float(np.average(((errores_sint[g][mascara] + c) ** 2).mean(axis=1), weights=pesos_ev))

    mejor_g = min(GRILLA_SINT, key=coste)
    return mejor_g, -np.average(errores_sint[mejor_g][mascara], axis=0, weights=pesos_ev)


ajustes_ev = {ev: ajustar_sintetico((obs_sint.evento != ev).to_numpy()) for ev in obs_sint.evento.unique()}
sint_ev = correr_sinteticos([(ev, int(f.Index), float(f.m), float(f.dist_hipo_km), int(500 + f.Index), *g_ev, [float(x) for x in c_ev])
                             for ev, (g_ev, c_ev) in ajustes_ev.items() for f in obs_sint[obs_sint.evento == ev].itertuples()])
filas_val = []
for ev, (g_ev, c_ev) in ajustes_ev.items():
    prueba = (obs_sint.evento == ev).to_numpy()
    corregido = np.array([sint_ev[(ev, int(j))] for j in obs_sint.index[prueba]]) - y_sint[prueba]
    filas_val.append({"evento": ev, "delta_sigma": g_ev[0], "kappa": g_ev[1], "q0": g_ev[2], "rmse_ln": rmse(corregido), "sesgo_ln": float(corregido.mean()),
                      "sesgo_ln_t01": float(corregido[:, 1].mean()), "sesgo_ln_t10": float(corregido[:, 5].mean())})
sintetico_val = pd.DataFrame(filas_val)
display(sintetico_val.round(3))
for f in sintetico_val.itertuples():
    registrar_modelo(f"acelerograma_sintetico_sin_{f.evento}", {"delta_sigma": f.delta_sigma, "kappa": f.kappa, "q0": f.q0},
                     {"rmse_ln": f.rmse_ln, "sesgo_ln": f.sesgo_ln, "sesgo_ln_t01": f.sesgo_ln_t01, "sesgo_ln_t10": f.sesgo_ln_t10})
PARAM_SINT, C_SINT = ajustar_sintetico(np.ones(len(obs_sint), bool))
registrar_modelo("acelerograma_sintetico", {"delta_sigma": PARAM_SINT[0], "kappa": PARAM_SINT[1], "q0": PARAM_SINT[2], "validacion": "evento excluido", "correccion": dict(zip(PERIODOS_CAL.tolist(), C_SINT.round(3).tolist()))},
                 {"rmse_ln_evento_excluido": float(np.sqrt((sintetico_val.rmse_ln ** 2).mean())), "sesgo_ln_max": float(sintetico_val.sesgo_ln.abs().max())})
ACELEROGRAMA_SINTETICO = {"delta_sigma": PARAM_SINT[0], "kappa": PARAM_SINT[1], "q0": PARAM_SINT[2], "periodos_cal": PERIODOS_CAL.tolist(),
                          "correccion": C_SINT.tolist(), "fs_hz": FS_SIM, "rho_g_cm3": RHO_G_CM3, "beta_km_s": BETA_KM_S}

## 15. Variables seleccionadas, parámetros de la cadena y métricas

In [ ]:
# Guarda variables seleccionadas, parámetros de la cadena y métricas.
(spark.createDataFrame(filas_sel, "caso string, columna string, tipo string, lote string")
 .write.mode("overwrite").option("replaceWhere", "caso IN ('dano', 'clustering')").saveAsTable("fe_columnas_seleccionadas"))

parametros = {"version_amplificacion": VERSION_AMP, "algoritmo_amplificacion": ALGORITMO_AMP, "pga_ref_nl": PGA_REF_NL,
              "sigma_f_sitio": SIGMA_F_SITIO, "sigma_loso": SIGMA_LOSO, "vs30_entreno": list(VS30_ENTRENO),
              "cobertura_km": COBERTURA_KM, "lote_clustering": lote_clustering,
              "acelerograma_sintetico": ACELEROGRAMA_SINTETICO}
(spark.createDataFrame([(k, json.dumps(v)) for k, v in parametros.items()], "clave string, valor string")
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("riesgo_parametros"))
guardar_metricas("07_demanda_sismica")